# ACS S2701 exploratory data analysis
Prepared by Thant Win.

**Status: the Python analysis pipeline completed on actual Census data.** Both five-year snapshots, output tables, and draft charts are included. Rerun the notebook to inspect the calculations; the notebook itself has not been executed in your Databricks workspace. See README.md and SPONSOR_MEETING_BRIEF.md.

## 1. Question and scope
Compare uninsured percentages across DE, DC, MD, PA, and VA in 2020–2024; explore age/sex differences and descriptive change from 2015–2019. These are aggregate estimates for the civilian noninstitutionalized population, not patient records or individual sample observations.

In [ ]:
from pathlib import Path
import sys, platform, importlib
PACKAGE_DIR = Path.cwd()  # Change to the directory containing s2701_eda.py in Databricks.
assert (PACKAGE_DIR / 's2701_eda.py').exists(), 'Set PACKAGE_DIR to the extracted package directory.'
sys.path.insert(0, str(PACKAGE_DIR))
import pandas as pd, numpy as np, matplotlib
print({'Python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__, 'matplotlib': matplotlib.__version__})
import s2701_eda as eda


## 2. Verify data definitions
The codebook records 240 variables across two vintages: six population groups × five measures × four fields (estimate, margin of error, and annotations) × two periods. These are metadata entries, not observed data rows. Actual estimates are included in data/.

In [ ]:
dictionary = pd.read_csv(PACKAGE_DIR / 'data_dictionary.csv')
display(dictionary.head(20))


## 3. Load and validate inputs
Put official ZIP/flat CSV downloads in data/. The loader requires all five jurisdictions and both periods. Alternatively configure CENSUS_API_KEY privately using your environment or Databricks secrets. Do not paste secrets into submitted notebooks.

In [ ]:
df = eda.prepare()
display(df[['vintage','period','geography','group','population','uninsured_pct','uninsured_pct_moe90']])


## 4. Missingness, composition, and source annotations
Review quality checks before interpreting results. Missing values and negative Census sentinel codes are retained as missing numeric values; raw values and annotations remain available. Blank annotation fields are expected for many ordinary values.

In [ ]:
total, change, quality = eda.analyze(df)
display(quality)
display(total[['period','geography','population','uninsured_count','uninsured_pct','uninsured_pct_moe90']])


## 5. Descriptive comparisons
Changes are in percentage points. This notebook does not perform significance testing, causal inference, or claim that adjacent ranks are statistically distinguishable. Cross-geography unweighted means are not regional population rates. Review source definitions before publishing a temporal comparison.

In [ ]:
display(change)
display(pd.read_csv(eda.OUT / 'jurisdiction_summary_unweighted.csv'))
display(pd.read_csv(eda.OUT / 'pooled_descriptive_rate.csv'))


## 6. Visualization examples
The first five plots illustrate the assigned comparison charts; CSV outputs provide table alternatives. The age comparison is optional. Error bars use published 90% margins of error. Inspect chart labels and accessibility before publication; all chart examples remain draft designs.

In [ ]:
eda.charts(df, total, change)
from IPython.display import Image, display
for chart in sorted(eda.OUT.glob('*.png')):
    print(chart.name)
    display(Image(filename=str(chart)))



## References
See the APA reference list and sources in SPONSOR_MEETING_BRIEF.md. Official metadata source URLs are https://api.census.gov/data/2019/acs/acs5/subject/groups/S2701.json and https://api.census.gov/data/2024/acs/acs5/subject/groups/S2701.json.